In [1]:
import os
import kagglehub
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. تحميل وتجهيز البيانات
path = kagglehub.dataset_download("rohitudageri/credit-card-details")
csv_file = None
for root, dirs, files in os.walk(path):
  for file in files:
    if file.endswith('.csv'):
      csv_file = os.path.join(root, file)
      break

df = pd.read_csv(csv_file)
df = df.drop(columns=[c for c in ['Ind_ID', 'ID', 'id'] if c in df.columns])
df = df.fillna(df.mode().iloc[0])

target_col = df.columns[-1]
for col in df.select_dtypes(include=['object']).columns:
  df[col] = LabelEncoder().fit_transform(df[col])

df[target_col] = (df[target_col] > 0).astype(int)

X = df.drop(columns=[target_col]).values
y = df[target_col].values.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))


# 2. الدوال الأساسية
def sigmoid(z):
  return 1 / (1 + np.exp(-np.clip(z, -500, 500)))


def compute_loss(y_true, y_pred):
  eps = 1e-15
  y_pred = np.clip(y_pred, eps, 1 - eps)
  return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


def compute_metrics(y_true, y_pred_prob):
  y_pred = (y_pred_prob >= 0.5).astype(int)
  TP = np.sum((y_pred == 1) & (y_true == 1))
  TN = np.sum((y_pred == 0) & (y_true == 0))
  FP = np.sum((y_pred == 1) & (y_true == 0))
  FN = np.sum((y_pred == 0) & (y_true == 1))

  acc = (TP + TN) / len(y_true)
  prec = TP / (TP + FP) if (TP + FP) > 0 else 0
  rec = TP / (TP + FN) if (TP + FN) > 0 else 0
  f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
  return acc, prec, rec, f1


# 3. تدريب النموذج باستخدام RMSprop من الصفر
np.random.seed(42)
w = np.zeros((X_train.shape[1], 1))
s = np.zeros_like(w)  # المتغير الخاص بتخزين مربعات الـ Gradients
lr = 0.01
beta = 0.9
epsilon = 1e-8
epochs = 150
n_samples = X_train.shape[0]

for epoch in range(epochs):
  z = np.dot(X_train, w)
  y_pred = sigmoid(z)
  dw = np.dot(X_train.T, (y_pred - y_train)) / n_samples

  # معادلة تحديث الـ RMSprop
  s = beta * s + (1 - beta) * (dw**2)
  w -= (lr / (np.sqrt(s) + epsilon)) * dw

# 4. التقييم وعرض النتائج
test_pred = sigmoid(np.dot(X_test, w))
loss = compute_loss(y_test, test_pred)
acc, prec, rec, f1 = compute_metrics(y_test, test_pred)

print('--- RMSprop Optimizer Results ---')
print(f'Loss      : {loss:.4f}')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1-Score  : {f1:.4f}')

--- RMSprop Optimizer Results ---
Loss      : 0.1999
Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1-Score  : 1.0000
